# Cryptocurrency Historical Data — Full Time-Series EDA

Comprehensive EDA for `dbx_joshdevph_dev.processed.cg_coin_historical_chart_data`.

The source `timestamp` is stored as **STRING**. This notebook first converts it to a true Spark timestamp and preserves the original value as `timestamp_raw`.

Coverage includes data quality, cadence/gaps, distributions, returns, volatility, outliers, rolling statistics, correlations, autocorrelation, lag relationships, and charts.

> This notebook does not overwrite the source table.


## Display Policy

To keep notebook output compact, all tabular `display()` outputs are limited to **10 rows**. Aggregate tables with fewer than 10 rows are unaffected in practice. Charts continue to use the full relevant time series so the visual analysis is not truncated.


## 1. Setup


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt

SOURCE_TABLE = "dbx_joshdevph_dev.processed.cg_coin_historical_chart_data"
EXPECTED_INTERVAL_MINUTES = 5
ROLLING_WINDOW = 12

df_raw = spark.table(SOURCE_TABLE)
print(f"Rows: {df_raw.count():,}")
df_raw.printSchema()


## 2. Parse the String Timestamp

Do not cast a date string such as `2026-08-18 16:00:00` directly to `long`. Parse it into a Spark timestamp first. `try_to_timestamp` returns NULL rather than failing when a malformed value is encountered.


In [0]:
df = (
    df_raw
    .withColumnRenamed("timestamp", "timestamp_raw")
    .withColumn(
        "timestamp",
        F.expr("try_to_timestamp(timestamp_raw, 'yyyy-MM-dd HH:mm:ss')")
    )
)

display(df.select(
        "coin_id", "vs_currency", "timestamp_raw", "timestamp",
        "price", "market_cap", "total_volume"
    ).limit(10))


In [0]:
display((df.agg(
        F.count("*").alias("rows"),
        F.sum(F.col("timestamp_raw").isNull().cast("int")).alias("null_raw_timestamp"),
        F.sum(
            (F.col("timestamp_raw").isNotNull() & F.col("timestamp").isNull()).cast("int")
        ).alias("failed_timestamp_parses")
    )).limit(10))


## 3. Dataset Coverage


In [0]:
coverage_df = (
    df.groupBy("coin_id", "vs_currency")
    .agg(
        F.count("*").alias("observations"),
        F.countDistinct("timestamp").alias("unique_timestamps"),
        F.min("timestamp").alias("start_timestamp"),
        F.max("timestamp").alias("end_timestamp")
    )
    .orderBy("coin_id", "vs_currency")
)
display((coverage_df).limit(10))


## 4. Missing Values and Duplicates


In [0]:
cols = ["coin_id", "vs_currency", "timestamp", "price", "market_cap", "total_volume"]
display((df.select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols])).limit(10))


In [0]:
duplicates_df = (
    df.groupBy("coin_id", "vs_currency", "timestamp")
    .count()
    .filter(F.col("count") > 1)
)
print(f"Duplicate keys: {duplicates_df.count():,}")
display((duplicates_df.orderBy("coin_id", "timestamp")).limit(10))


## 5. Sampling Cadence and Time Gaps

This fixes the original `CAST_INVALID_INPUT` error by calculating differences only after timestamp parsing.


In [0]:
time_window = Window.partitionBy("coin_id", "vs_currency").orderBy("timestamp")

df_ts = (
    df.filter(F.col("timestamp").isNotNull())
    .withColumn("previous_timestamp", F.lag("timestamp").over(time_window))
    .withColumn(
        "interval_minutes",
        (
            F.unix_timestamp("timestamp")
            - F.unix_timestamp("previous_timestamp")
        ) / 60.0
    )
)

display(df_ts.select("coin_id", "timestamp", "previous_timestamp", "interval_minutes")
    .orderBy("coin_id", "timestamp")
    .limit(20))


In [0]:
display((df_ts.filter(F.col("interval_minutes").isNotNull())
    .groupBy("coin_id", "vs_currency")
    .agg(
        F.count("*").alias("intervals"),
        F.avg("interval_minutes").alias("mean_minutes"),
        F.expr("percentile_approx(interval_minutes, 0.5)").alias("median_minutes"),
        F.min("interval_minutes").alias("min_minutes"),
        F.max("interval_minutes").alias("max_minutes"),
        F.sum((F.col("interval_minutes") > EXPECTED_INTERVAL_MINUTES).cast("int"))
         .alias("intervals_above_expected")
    )).limit(10))


### Largest Time Gaps


In [0]:
display((df_ts.filter(F.col("interval_minutes") > EXPECTED_INTERVAL_MINUTES)
    .select("coin_id", "previous_timestamp", "timestamp", "interval_minutes")
    .orderBy(F.desc("interval_minutes"))).limit(10))


## 6. Descriptive Statistics


In [0]:
display((df.groupBy("coin_id", "vs_currency")
    .agg(
        F.count("*").alias("n"),
        F.avg("price").alias("mean_price"),
        F.stddev_samp("price").alias("std_price"),
        F.min("price").alias("min_price"),
        F.expr("percentile_approx(price, 0.25)").alias("price_q1"),
        F.expr("percentile_approx(price, 0.5)").alias("price_median"),
        F.expr("percentile_approx(price, 0.75)").alias("price_q3"),
        F.max("price").alias("max_price"),
        F.avg("market_cap").alias("mean_market_cap"),
        F.stddev_samp("market_cap").alias("std_market_cap"),
        F.avg("total_volume").alias("mean_volume"),
        F.stddev_samp("total_volume").alias("std_volume")
    )).limit(10))


## 7. Historical Price Charts


In [0]:
coins = [r["coin_id"] for r in df.select("coin_id").distinct().orderBy("coin_id").collect()]

for coin_id in coins:
    pdf = (
        df.filter(F.col("coin_id") == coin_id)
        .select("timestamp", "price").orderBy("timestamp").toPandas()
    )
    plt.figure(figsize=(14, 5))
    plt.plot(pdf["timestamp"], pdf["price"])
    plt.title(f"{coin_id.title()} — Historical Price")
    plt.xlabel("Timestamp")
    plt.ylabel("Price")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## 8. Market Capitalization Charts


In [0]:
for coin_id in coins:
    pdf = df.filter(F.col("coin_id") == coin_id).select("timestamp", "market_cap").orderBy("timestamp").toPandas()
    plt.figure(figsize=(14, 5))
    plt.plot(pdf["timestamp"], pdf["market_cap"])
    plt.title(f"{coin_id.title()} — Market Capitalization")
    plt.xlabel("Timestamp")
    plt.ylabel("Market Cap")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## 9. Trading Volume Charts


In [0]:
for coin_id in coins:
    pdf = df.filter(F.col("coin_id") == coin_id).select("timestamp", "total_volume").orderBy("timestamp").toPandas()
    plt.figure(figsize=(14, 5))
    plt.plot(pdf["timestamp"], pdf["total_volume"])
    plt.title(f"{coin_id.title()} — Trading Volume")
    plt.xlabel("Timestamp")
    plt.ylabel("Total Volume")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## 10. Price Changes and Returns

`Return (%) = (Current Price / Previous Price - 1) × 100`

Returns make volatility more comparable across coins with different price levels.


In [0]:
df_returns = (
    df.filter(F.col("timestamp").isNotNull())
    .withColumn("previous_price", F.lag("price").over(time_window))
    .withColumn("price_change", F.col("price") - F.col("previous_price"))
    .withColumn("return_pct", ((F.col("price") / F.col("previous_price")) - 1) * 100)
)

display(df_returns.select(
        "coin_id", "timestamp", "price", "previous_price", "price_change", "return_pct"
    ).orderBy("coin_id", "timestamp").limit(20))


In [0]:
display((df_returns.filter(F.col("return_pct").isNotNull())
    .groupBy("coin_id", "vs_currency")
    .agg(
        F.avg("return_pct").alias("mean_return_pct"),
        F.stddev_samp("return_pct").alias("return_std_pct"),
        F.min("return_pct").alias("min_return_pct"),
        F.expr("percentile_approx(return_pct, 0.25)").alias("q1"),
        F.expr("percentile_approx(return_pct, 0.5)").alias("median"),
        F.expr("percentile_approx(return_pct, 0.75)").alias("q3"),
        F.max("return_pct").alias("max_return_pct")
    )).limit(10))


## 11. Returns Over Time


In [0]:
for coin_id in coins:
    pdf = (
        df_returns.filter((F.col("coin_id") == coin_id) & F.col("return_pct").isNotNull())
        .select("timestamp", "return_pct").orderBy("timestamp").toPandas()
    )
    plt.figure(figsize=(14, 5))
    plt.plot(pdf["timestamp"], pdf["return_pct"])
    plt.axhline(0, linewidth=1)
    plt.title(f"{coin_id.title()} — Returns Over Time")
    plt.xlabel("Timestamp")
    plt.ylabel("Return (%)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## 12. Price and Return Distributions


In [0]:
for coin_id in coins:
    pdf = df.filter(F.col("coin_id") == coin_id).select("price").dropna().toPandas()
    plt.figure(figsize=(9, 5))
    plt.hist(pdf["price"], bins=40)
    plt.title(f"{coin_id.title()} — Price Distribution")
    plt.xlabel("Price")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [0]:
for coin_id in coins:
    pdf = (
        df_returns.filter((F.col("coin_id") == coin_id) & F.col("return_pct").isNotNull())
        .select("return_pct").toPandas()
    )
    plt.figure(figsize=(9, 5))
    plt.hist(pdf["return_pct"], bins=50)
    plt.title(f"{coin_id.title()} — Return Distribution")
    plt.xlabel("Return (%)")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.boxplot(pdf["return_pct"], vert=False)
    plt.title(f"{coin_id.title()} — Return Boxplot")
    plt.xlabel("Return (%)")
    plt.tight_layout()
    plt.show()


## 13. IQR-Based Return Outliers


In [0]:
return_bounds_df = (
    df_returns.filter(F.col("return_pct").isNotNull())
    .groupBy("coin_id", "vs_currency")
    .agg(
        F.expr("percentile_approx(return_pct, 0.25)").alias("q1"),
        F.expr("percentile_approx(return_pct, 0.75)").alias("q3")
    )
    .withColumn("iqr", F.col("q3") - F.col("q1"))
    .withColumn("lower_bound", F.col("q1") - 1.5 * F.col("iqr"))
    .withColumn("upper_bound", F.col("q3") + 1.5 * F.col("iqr"))
)

return_outliers_df = (
    df_returns.join(return_bounds_df, ["coin_id", "vs_currency"], "left")
    .filter((F.col("return_pct") < F.col("lower_bound")) | (F.col("return_pct") > F.col("upper_bound")))
)

display((return_outliers_df.select(
        "coin_id", "timestamp", "price", "return_pct", "lower_bound", "upper_bound"
    ).orderBy(F.desc(F.abs("return_pct")))).limit(10))


## 14. Rolling Price and Volatility


In [0]:
rolling_window = (
    Window.partitionBy("coin_id", "vs_currency")
    .orderBy("timestamp")
    .rowsBetween(-(ROLLING_WINDOW - 1), 0)
)

df_rolling = (
    df_returns
    .withColumn("rolling_price_mean", F.avg("price").over(rolling_window))
    .withColumn("rolling_price_std", F.stddev_samp("price").over(rolling_window))
    .withColumn("rolling_return_std", F.stddev_samp("return_pct").over(rolling_window))
)


In [0]:
for coin_id in coins:
    pdf = (
        df_rolling.filter(F.col("coin_id") == coin_id)
        .select("timestamp", "price", "rolling_price_mean").orderBy("timestamp").toPandas()
    )
    plt.figure(figsize=(14, 5))
    plt.plot(pdf["timestamp"], pdf["price"], label="Price")
    plt.plot(pdf["timestamp"], pdf["rolling_price_mean"], label=f"{ROLLING_WINDOW}-period rolling mean")
    plt.title(f"{coin_id.title()} — Price and Rolling Mean")
    plt.xlabel("Timestamp")
    plt.ylabel("Price")
    plt.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


In [0]:
for coin_id in coins:
    pdf = (
        df_rolling.filter(F.col("coin_id") == coin_id)
        .select("timestamp", "rolling_return_std").orderBy("timestamp").toPandas()
    )
    plt.figure(figsize=(14, 5))
    plt.plot(pdf["timestamp"], pdf["rolling_return_std"])
    plt.title(f"{coin_id.title()} — Rolling Return Volatility")
    plt.xlabel("Timestamp")
    plt.ylabel("Rolling Std. Dev. of Returns (%)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## 15. Correlations and Price vs Volume


In [0]:
display((df.groupBy("coin_id", "vs_currency")
    .agg(
        F.corr("price", "market_cap").alias("price_market_cap_corr"),
        F.corr("price", "total_volume").alias("price_volume_corr"),
        F.corr("market_cap", "total_volume").alias("market_cap_volume_corr")
    )).limit(10))


In [0]:
for coin_id in coins:
    pdf = df.filter(F.col("coin_id") == coin_id).select("total_volume", "price").dropna().toPandas()
    plt.figure(figsize=(8, 6))
    plt.scatter(pdf["total_volume"], pdf["price"], alpha=0.5)
    plt.title(f"{coin_id.title()} — Price vs Trading Volume")
    plt.xlabel("Total Volume")
    plt.ylabel("Price")
    plt.tight_layout()
    plt.show()


## 16. Time-Series Dependence: Autocorrelation at Model Lags

This checks the same lags currently used by the forecasting feature pipeline: 1, 2, 3, 6, and 12 observations.


In [0]:
lag_values = [1, 2, 3, 6, 12]
lagged_df = df.filter(F.col("timestamp").isNotNull())

for lag in lag_values:
    lagged_df = lagged_df.withColumn(
        f"price_lag_{lag}",
        F.lag("price", lag).over(time_window)
    )

display((lagged_df.groupBy("coin_id", "vs_currency")
    .agg(*[
        F.corr("price", f"price_lag_{lag}").alias(f"price_acf_lag_{lag}")
        for lag in lag_values
    ])).limit(10))


In [0]:
returns_lagged_df = df_returns.filter(F.col("return_pct").isNotNull())

for lag in lag_values:
    returns_lagged_df = returns_lagged_df.withColumn(
        f"return_lag_{lag}",
        F.lag("return_pct", lag).over(time_window)
    )

display((returns_lagged_df.groupBy("coin_id", "vs_currency")
    .agg(*[
        F.corr("return_pct", f"return_lag_{lag}").alias(f"return_acf_lag_{lag}")
        for lag in lag_values
    ])).limit(10))


## 17. Lag-1 Price Relationship


In [0]:
for coin_id in coins:
    pdf = (
        lagged_df.filter((F.col("coin_id") == coin_id) & F.col("price_lag_1").isNotNull())
        .select("price_lag_1", "price").toPandas()
    )
    plt.figure(figsize=(7, 6))
    plt.scatter(pdf["price_lag_1"], pdf["price"], alpha=0.5)
    plt.title(f"{coin_id.title()} — Previous Price vs Current Price")
    plt.xlabel("Price Lag 1")
    plt.ylabel("Current Price")
    plt.tight_layout()
    plt.show()


## 18. Largest Absolute Returns


In [0]:
display((df_returns.filter(F.col("return_pct").isNotNull())
    .withColumn("absolute_return_pct", F.abs("return_pct"))
    .select(
        "coin_id", "timestamp", "price", "previous_price",
        "price_change", "return_pct", "absolute_return_pct", "total_volume"
    )
    .orderBy(F.desc("absolute_return_pct"))).limit(10))


## 19. EDA Interpretation Checklist

### Data integrity
- Did all text timestamps parse successfully?
- Are there duplicate coin/currency/timestamp rows?
- Are price, market cap, or volume values missing?
- Is the observation cadence actually five minutes?
- Are there meaningful time gaps?

### Time-series behavior
- Is there a visible trend or regime change?
- Does volatility change through time?
- Are returns heavy-tailed or highly skewed?
- Are extreme movements genuine market events or data issues?
- Is volatility clustered?

### Forecasting implications
- How strongly is current price related to lagged prices?
- Do returns show meaningful short-lag autocorrelation?
- Are the two coins materially different in volatility?
- Are the train and future periods likely to represent similar regimes?

### Important lag-feature consideration

Your current feature engineering uses `LAG(price, n)`. This means **n observations ago**, not necessarily **n × 5 minutes ago**.

If this EDA finds irregular or missing intervals, `price_lag_12` may not always mean exactly 60 minutes. If exact time-based lags matter, consider regularizing each coin/currency time series to a five-minute grid before generating lag features.
